# NB2 — Hybrid Search: BM25 + Vector + RRF

**Stack:** BM25 + bge-m3 + Qdrant Docker; dùng collection `lab19` của NB1.
Mục tiêu: so sánh Precision@10 trên cùng 50 golden queries.
BM25 tìm theo từ khóa, vector tìm theo ý nghĩa, RRF kết hợp thứ hạng.
Kết quả phụ thuộc corpus và model; chỉ kết luận sau khi đo.

Trước khi chạy: lưu NB1 và shutdown kernel NB1 để giải phóng model trong RAM.
Chọn kernel **Python (Lab 19)** cho notebook này.

In [1]:
import _setup  # noqa: F401
import json
import os
import statistics
from collections import defaultdict
from functools import lru_cache
from pathlib import Path

from app.evaluation import precision_at_k, save_result
from app.fusion import KEYWORD_WEIGHT, SEMANTIC_WEIGHT, fuse_rankings
from app.search import COLLECTION, Searcher

DATA = Path(_setup.__file__).resolve().parent.parent / "data"

D:\AI_Thucchien\K4-Track2-Day19-VectorFeatureStore-Lab\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Đọc corpus, dựng BM25 và dùng lại vector NB1

BM25 chỉ cần token hóa văn bản. Vector đã lưu trong Qdrant được kiểm tra
số chiều, số tài liệu và payload trước khi dùng; cell này không embed lại corpus.

In [2]:
assert os.getenv("QDRANT_MODE") == "server", "Kiểm tra .env và restart kernel"
searcher = Searcher.from_corpus(DATA / "corpus_vn.jsonl", require_existing=True)
docs = searcher.docs
bm25, embedder, client = searcher.bm25, searcher.embedder, searcher.client
assert len(docs) == 1000
assert embedder.backend == "bge-m3"
assert searcher.reused_index
print(f"Corpus: {len(docs)} docs")
print(f"Model: {embedder.model_name} ({embedder.dim} chiều)")
print(f"Collection: {COLLECTION} — dùng lại vector từ NB1")
print("BM25 + vector indices ready")
print("Ignored common corpus tokens:", sorted(searcher.bm25_ignored_tokens))

Corpus: 1000 docs
Model: BAAI/bge-m3 (1024 chiều)
Collection: lab19 — dùng lại vector từ NB1
BM25 + vector indices ready
Ignored common corpus tokens: ['biến', 'bài', 'các', 'cách', 'công', 'cụ', 'dụ', 'giải', 'gồm', 'hoạ', 'hỏi', 'khái', 'kèm', 'là', 'minh', 'một', 'niệm', 'này', 'nền', 'phải', 'phổ', 'quan', 'thích', 'thực', 'trọng', 'tảng', 'tế', 'viết', 'và', 'ví', 'với', 'đòi']


## 2. Hai hàm tìm kiếm cơ sở

Mỗi hàm trả danh sách doc_id theo thứ hạng. Khi kết hợp RRF, lấy top-50
từ mỗi bộ tìm kiếm trước khi chọn top-10.
Cache vector câu hỏi giúp tránh embed cùng câu hai lần trong phép đo chất lượng.
API ở NB3 vẫn embed từng request; cache này chỉ dùng trong notebook NB2.

In [3]:
TOP_K = 10
RRF_K = 60

@lru_cache(maxsize=256)
def search_keyword(query: str, top_k: int = TOP_K) -> list[str]:
    tokens = [token for token in searcher._tokenize(query)
              if token not in searcher.bm25_ignored_tokens]
    scores = bm25.get_scores(tokens)
    # Only documents with positive lexical evidence participate in ranking.
    candidates = [i for i, score in enumerate(scores) if score > 0]
    ranked = sorted(candidates, key=lambda i: -scores[i])[:top_k]
    return [docs[i]["doc_id"] for i in ranked]

@lru_cache(maxsize=128)
def query_vector(query: str) -> tuple[float, ...]:
    return tuple(next(embedder.embed([query])).tolist())

@lru_cache(maxsize=256)
def search_semantic(query: str, top_k: int = TOP_K) -> list[str]:
    result = client.query_points(
        collection_name=COLLECTION,
        query=list(query_vector(query)),
        limit=top_k,
    )
    return [point.payload["doc_id"] for point in result.points]

## 3. Weighted Reciprocal Rank Fusion

Standard RRF: sum of 1 / (k + rank), with one-based ranks and k=60.
Weighted RRF: 0.2 / (k + keyword_rank) + 0.8 / (k + semantic_rank).
A document absent from one ranking receives no vote from that source.
Semantic gets more influence because it outperformed BM25 in the previous run.
Both variants are reported. This adjustment followed inspection of the lab
golden set, so these results are exploratory, not independent held-out scores.

In [4]:
def search_hybrid(
    query: str, top_k: int = TOP_K, rrf_k: int = RRF_K,
    keyword_weight: float = KEYWORD_WEIGHT, semantic_weight: float = SEMANTIC_WEIGHT,
) -> list[str]:
    depth = max(top_k * 5, 50)
    fused = fuse_rankings(
        search_keyword(query, depth), search_semantic(query, depth), k=rrf_k,
        keyword_weight=keyword_weight, semantic_weight=semantic_weight,
    )
    return [doc_id for doc_id, _ in fused[:top_k]]

print(f"Weighted RRF: keyword={KEYWORD_WEIGHT}, semantic={SEMANTIC_WEIGHT}, k={RRF_K}")

test_q = "co giãn linh hoạt theo nhu cầu sử dụng"
print(f"Query: {test_q}")
for mode, fn in (("keyword", search_keyword), ("semantic", search_semantic), ("hybrid", search_hybrid)):
    print(f"  {mode:8} top-3: {fn(test_q)[:3]}")

Weighted RRF: keyword=0.2, semantic=0.8, k=60
Query: co giãn linh hoạt theo nhu cầu sử dụng
  keyword  top-3: ['mobile_082', 'mobile_093', 'mobile_092']
  semantic top-3: ['cloud_005', 'mobile_020', 'mobile_039']
  hybrid   top-3: ['mobile_093', 'cloud_072', 'mobile_025']


## 4. Đánh giá trên 50 golden queries

Precision@10 = số tài liệu relevant trong top-10 / 10.
Dùng relevant_doc_ids của golden set; cùng query và cùng corpus cho ba mode.
Cell in tiến độ, bảng trung bình và kiểm tra mục tiêu hybrid thắng cả hai mode.

In [5]:
with (DATA / "golden_set.jsonl").open(encoding="utf-8") as f:
    golden = [json.loads(line) for line in f if line.strip()]
assert len(golden) == 50

p_kw, p_sem, p_hyb, p_equal_rrf, query_results = [], [], [], [], []
for i, q in enumerate(golden, start=1):
    relevant = set(q["relevant_doc_ids"])
    scores = {
        "keyword": precision_at_k(search_keyword(q["query"]), relevant, TOP_K),
        "semantic": precision_at_k(search_semantic(q["query"]), relevant, TOP_K),
        "hybrid": precision_at_k(search_hybrid(q["query"]), relevant, TOP_K),
    }
    equal_rrf_score = precision_at_k(
        search_hybrid(q["query"], keyword_weight=1.0, semantic_weight=1.0), relevant, TOP_K,
    )
    p_equal_rrf.append(equal_rrf_score)
    p_kw.append(scores["keyword"])
    p_sem.append(scores["semantic"])
    p_hyb.append(scores["hybrid"])
    query_results.append({
        "query_id": q["query_id"], "query": q["query"],
        "type": q["mode_hint"], "topic": q["topic"],
        "hybrid_equal_rrf": equal_rrf_score, **scores,
    })
    if i % 10 == 0:
        print(f"Evaluated: {i}/{len(golden)} queries")

averages = {
    "keyword": statistics.mean(p_kw),
    "semantic": statistics.mean(p_sem),
    "hybrid": statistics.mean(p_hyb),
}
print("\nPrecision@10 — trung bình 50 queries")
print(f"  {'mode':10} {'Precision@10':>14}")
for mode, value in averages.items():
    print(f"  {mode:10} {value:>13.1%}")
print(f"  Equal RRF baseline: {statistics.mean(p_equal_rrf):.1%}")
hybrid_beats_both = averages["hybrid"] > max(averages["keyword"], averages["semantic"])
print("PASS — hybrid thắng cả keyword và semantic" if hybrid_beats_both
      else "Chưa đạt mục tiêu hybrid thắng cả hai; giữ số đo để phân tích.")

Evaluated: 10/50 queries
Evaluated: 20/50 queries
Evaluated: 30/50 queries
Evaluated: 40/50 queries
Evaluated: 50/50 queries

Precision@10 — trung bình 50 queries
  mode         Precision@10
  keyword            80.4%
  semantic           95.2%
  hybrid             95.0%
  Equal RRF baseline: 90.0%
Chưa đạt mục tiêu hybrid thắng cả hai; giữ số đo để phân tích.


## 5. So sánh theo loại query và lưu kết quả

exact: có từ khóa rõ; paraphrase: diễn đạt cùng ý bằng từ khác;
mixed: kết hợp từ khóa và diễn đạt lại. Quan sát mode thắng ở từng nhóm.
File JSON lưu số đo thật để tạo reflection sau khi hoàn thành NB4.

In [6]:
by_type = defaultdict(lambda: defaultdict(list))
for q, scores in zip(golden, query_results):
    for mode in averages:
        by_type[q["mode_hint"]][mode].append(scores[mode])

slice_results = {}
print(f"  {'type':12} {'n':>3} {'keyword':>10} {'semantic':>10} {'hybrid':>10}")
for query_type in ("exact", "paraphrase", "mixed"):
    group = by_type[query_type]
    values = {mode: statistics.mean(group[mode]) for mode in averages}
    slice_results[query_type] = {"n": len(group["keyword"]), **values}
    print(f"  {query_type:12} {len(group['keyword']):>3} "
          f"{values['keyword']:>9.1%} {values['semantic']:>9.1%} {values['hybrid']:>9.1%}")

result_path = save_result("nb2_quality.json", {
    "n_queries": len(golden), "top_k": TOP_K, "rrf_k": RRF_K,
    "fusion_method": "weighted_rrf",
    "fusion_weights": {"keyword": KEYWORD_WEIGHT, "semantic": SEMANTIC_WEIGHT},
    "equal_rrf_average": statistics.mean(p_equal_rrf),
    "evaluation_note": "Adjusted after observing lab golden results; not an independent held-out evaluation",
    "embedding_backend": embedder.backend, "collection": COLLECTION,
    "keyword_min_score_exclusive": 0.0,
    "keyword_preprocessing": "unicode_tokens_corpus_df_below_80_percent",
    "keyword_ignored_tokens": sorted(searcher.bm25_ignored_tokens),
    "average": averages, "by_type": slice_results,
    "hybrid_beats_both": hybrid_beats_both, "queries": query_results,
})
print(f"Saved: {result_path}")

  type           n    keyword   semantic     hybrid
  exact         15    100.0%     98.0%     99.3%
  paraphrase    15     34.7%     86.7%     84.7%
  mixed         20    100.0%     99.5%     99.5%
Saved: D:\AI_Thucchien\K4-Track2-Day19-VectorFeatureStore-Lab\submission\results\nb2_quality.json


## Bằng chứng cần lưu

- Mục 4: bảng Precision@10 và kết luận từ số đo.
- Mục 5: bảng exact/paraphrase/mixed.
- Ctrl+S để giữ output notebook.
- Ảnh: `submission/screenshots/nb2_precision.png` và `nb2_slices.png`.

Khi hybrid chưa thắng: đối chiếu từng query trong nb2_quality.json và các
danh sách top-10, kiểm tra RRF, chất lượng BM25 và model. Giữ nguyên số đo.

## 6. Chẩn đoán: cấu hình nào có thể vượt Semantic?

Chạy cell này sau mục 5. So sánh **25 cấu hình**: độ sâu ứng viên
10/20/50/100/200 và trọng số keyword 0.05/0.1/0.2/0.35/0.5.
Trọng số semantic = 1 - trọng số keyword; mọi cấu hình đều giữ k=60.
Mỗi query chỉ cần một vector đã cache; không embed lại corpus.

Bảng in số tài liệu đúng tăng/giảm so với Semantic, tính trên 50 × 10 kết quả.
Chỉ cấu hình có Precision@10 cao hơn cả hai baseline mới được ghi là thắng.
Khi bằng điểm, ưu tiên độ sâu nhỏ hơn để giảm chi phí truy vấn.

Đây là thử nghiệm trên chính golden set đã xem, chưa phải kiểm chứng trên dữ liệu mới.
Kết quả không tự đổi cấu hình API hay ghi đè số đo ở mục 5.
`candidate_ceiling` trong JSON chỉ là trần lý thuyết dựa trên nhãn để chẩn đoán;
không phải điểm đo của hệ thống và không được dùng để xếp hạng lúc tìm kiếm.

Cơ sở điều chỉnh: [Qdrant — tuning hybrid search](https://qdrant.tech/documentation/search-tuning/how-to-tune-hybrid-search/).
Công thức ở đây là tổng weight/(60+rank) của lab, được tính trong Python.

In [7]:
from scripts.analyze_hybrid import DEPTHS, analyze_rankings

ranking_records = []
for i, q in enumerate(golden, start=1):
    ranking_records.append({
        "query_id": q["query_id"], "query": q["query"], "type": q["mode_hint"],
        "relevant_doc_ids": q["relevant_doc_ids"],
        "keyword": search_keyword(q["query"], TOP_K),
        "semantic": search_semantic(q["query"], TOP_K),
        "rankings": {
            str(depth): {
                "keyword": search_keyword(q["query"], depth),
                "semantic": search_semantic(q["query"], depth),
            } for depth in DEPTHS
        },
    })
    if i % 10 == 0:
        print(f"Collected rankings: {i}/{len(golden)} queries")

rankings_path = save_result("nb2_rankings.json", {
    "embedding_backend": embedder.backend, "collection": COLLECTION,
    "keyword_preprocessing": "unicode_tokens_corpus_df_below_80_percent",
    "records": ranking_records,
})
tuning = analyze_rankings(ranking_records, k=RRF_K, top_k=TOP_K)
tuning_path = save_result("nb2_tuning.json", tuning)
print(f"\nKeyword: {tuning['baseline']['keyword']:.1%}; Semantic: {tuning['baseline']['semantic']:.1%}")
print(f"{'depth':>5} {'w_kw':>6} {'w_sem':>6} {'P@10':>7} {'delta docs':>10} {'win/loss/tie':>14} {'beats both':>11}")
for row in tuning["configurations"]:
    counts = f"{row['wins']}/{row['losses']}/{row['ties']}"
    print(f"{row['depth']:5} {row['keyword_weight']:6.2f} {row['semantic_weight']:6.2f} "
          f"{row['average']:6.1%} {row['correct_delta']:+10d} {counts:>14} {str(row['beats_both']):>11}")

best = tuning["configurations"][0]
print(f"\nBest measured configuration: depth={best['depth']}, "
      f"keyword={best['keyword_weight']}, semantic={best['semantic_weight']}; P@10={best['average']:.1%}")
print(f"Configurations beating both baselines: {tuning['winning_configurations']}/25")
print("Queries whose number of relevant top-10 documents changed versus Semantic:")
for row in best["queries"]:
    if row["correct_delta"]:
        print(f"  {row['query_id']} {row['correct_delta']:+d}: {row['query']}")
if not tuning["winning_configurations"]:
    print("No tested configuration beats both baselines. Do not claim a Hybrid win.")
print(f"Saved rankings: {rankings_path}")
print(f"Saved comparison: {tuning_path}")

Collected rankings: 10/50 queries
Collected rankings: 20/50 queries
Collected rankings: 30/50 queries
Collected rankings: 40/50 queries
Collected rankings: 50/50 queries

Keyword: 80.4%; Semantic: 95.2%
depth   w_kw  w_sem    P@10 delta docs   win/loss/tie  beats both
   50   0.05   0.95  95.4%         +1         1/0/49        True
   10   0.05   0.95  95.2%         +0         0/0/50       False
   10   0.10   0.90  95.2%         +0         0/0/50       False
   10   0.20   0.80  95.2%         +0         0/0/50       False
   10   0.35   0.65  95.2%         +0         0/0/50       False
   20   0.05   0.95  95.2%         +0         0/0/50       False
   20   0.10   0.90  95.2%         +0         1/1/48       False
   50   0.10   0.90  95.2%         +0         1/1/48       False
  100   0.05   0.95  95.2%         +0         1/1/48       False
  100   0.10   0.90  95.2%         +0         2/2/46       False
  100   0.20   0.80  95.2%         +0         2/4/44       False
  200   0.05   0